In [ ]:
import csv
import os
import tempfile
import time
from operator import attrgetter

import ansys.aedt.core
import math

# Define constants.
import pickle
AEDT_VERSION = "2025.1"
NUM_CORES = 4
NG_MODE = False  # Open AEDT UI when it is launched.


In [ ]:
import pickle
geodata={
"R":112.500000, #stator radius
"r":74.950000, #rotor radius
"p" :3,   #pole pair
"g" :0.700000,   #air gap thickness
"nlay" :1,   #layer of rotor barrier
"PM_CS":[[45.9407,60.2576,],[43.0557,18.2579,],[0.98942,0.62033,],[0.14506,0.78434,],], #Coordinate system magnet: 0-x 1-y 2-xvect 3-yvect
"q":3,  #slot per cave per phase
"radial_ribs_split":0, #area added by split barrier
"n_PM":2, #number of permanent magnet (number of magnetic segments)
"l":134.000000, #stack length
"filepath":"D:/syre working copy/motorExamples/", #motor"s folder
"filename":"TeslaModel3.mat", #motor"s file mat name
}
#struct material names
material={
"rotor":"M270-35A",
"stator":"M270-35A",
"shaft":"ShaftAir",
"slotcond":"Copper",
"magnet":"BMN-52UH",
}
with open('D:/KangDH/git_syRe/syreExport/syre_AnsysMaxwell/temp/temp.pkl', 'wb') as export:
    pickle.dump([geodata,material],export,protocol=2)

In [ ]:
filepath = r"D:\KangDH\git_syRe\syreExport\syre_AnsysMaxwell\temp\temp.pkl"
with open(filepath,'rb') as exportdata:
    geo,material = pickle.load(exportdata)

In [ ]:
geo

In [ ]:
dxfPath=r'D:\KangDH\git_syReTeslaModel3.dxf'

In [ ]:
m2d = ansys.aedt.core.Maxwell2d(
    version=AEDT_VERSION,
    new_desktop=False,
    non_graphical=NG_MODE,
)

# ## Define modeler units

m2d.modeler.model_units = "mm"

In [ ]:
curPjtName=m2d.project_name
curDesignName=m2d.design_name
listPjt=m2d.project_list
listDesign=m2d.design_list
m2d.set_active_design(curDesignName)


In [ ]:
m2d.oeditor

In [ ]:
m2dEditor=m2d.oeditor

## rotor

In [68]:
m2dEditor.ImportDXF(
	[
		"NAME:options",
		"FileName:="		, dxfPath,
		"Scale:="		, 0.001,
		"AutoDetectClosed:="	, True,
		"SelfStitch:="		, True,
		"DefeatureGeometry:="	, True,
		"DefeatureDistance:="	, 0.0001,
		"RoundCoordinates:="	, False,
		"RoundNumDigits:="	, 4,
		"SelfStitchTolerance:="	, 0.0001,
		"WritePolyWithWidthAsFilledPoly:=", True,
		"ImportMethod:="	, 1,
		"2DSheetBodies:="	, True,
		[
			"NAME:LayerInfo",
			[
				"NAME:0",
				"source:="		, "0",
				"display_source:="	, "0",
				"import:="		, True,
				"dest:="		, "0",
				"dest_selected:="	, False,
				"layer_type:="		, "signal"
			],
			[
				"NAME:1",
				"source:="		, "1",
				"display_source:="	, "1",
				"import:="		, True,
				"dest:="		, "1",
				"dest_selected:="	, False,
				"layer_type:="		, "signal"
			],
			[
				"NAME:2",
				"source:="		, "2",
				"display_source:="	, "2",
				"import:="		, True,
				"dest:="		, "2",
				"dest_selected:="	, False,
				"layer_type:="		, "signal"
			]
		]
	])


In [69]:
m2dEditor.FitAll()

In [71]:
####################### detach slot  ################
ii=0
slotconds=""
lineslotairgaps=""
slotairgaps=""
statplate="1_1"

for ii in range(geo["q"]*3):    
	slotindex=2+ii*4
	lineindex=slotindex+1
	slot="1_%d" %(slotindex)
	line="1_%d"%(lineindex)
	lineairgap="1_%d,1_%d"%(lineindex+1,lineindex+2)
	slotconds = slotconds + "," + slot + "," + slot + "_Detach1"   ##list slot/cond, useful for detachment
	lineslotairgaps=lineslotairgaps+","+lineairgap
	slotairgaps=slotairgaps+","+statplate+"_Detach"+"%d"%(ii+1)


	########## Slot
	m2dEditor.Imprint(
		[
			"NAME:Selections",
			"Blank Parts:="		, slot,
			"Tool Parts:="		, line
		], 
		[
			"NAME:ImprintParameters",
			"KeepOriginals:="	, False
		])

	facesIDs=m2dEditor.GetFaceIDs(slot)
	
	faceID=int(facesIDs[1])
	m2dEditor.DetachFaces(
		[
			"NAME:Selections",
			"Selections:="		, slot,
			"NewPartsModelFlag:="	, "Model"
		], 
		[
			"NAME:Parameters",
			[
				"NAME:DetachFacesToParameters",
				"FacesToDetach:="	, [faceID]
			]
		])

lineslotairgaps=lineslotairgaps[1:]

slotconds=slotconds[1:] #removing initial comma of the string
slotconds_array=slotconds.split(',') #vector containig the elements of the string separated by the comma

slotairgaps=slotairgaps[1:]
slotairgaps_array=slotairgaps.split(',')

############ detach slot from stator's iron ###############
m2dEditor.Subtract(
	[
		"NAME:Selections",
		"Blank Parts:="		, statplate,
		"Tool Parts:="		, slotconds
	], 
	[
		"NAME:SubtractParameters",
		"KeepOriginals:="	, True
	])
########### detach slot airgap from stator's iron


m2dEditor.Imprint(
	[
		"NAME:Selections",
		"Blank Parts:="		, statplate,
		"Tool Parts:="		, lineslotairgaps
	], 
	[
		"NAME:ImprintParameters",
		"KeepOriginals:="	, False
	])

facesIDs=m2dEditor.GetFaceIDs(statplate)
facesIDs=facesIDs[:-1] #remove last term
facesIDs=map(int,facesIDs) #array of string -> integer array


GrpcApiError: Failed to execute gRPC AEDT command: Imprint

In [72]:
facesIDs

In [73]:

m2dEditor.DetachFaces(
	[
		"NAME:Selections",
		"Selections:="		, statplate,
		"NewPartsModelFlag:="	, "Model"
	], 
	[
		"NAME:Parameters",
		[
			"NAME:DetachFacesToParameters",
			"FacesToDetach:="	, facesIDs
		]
	])


GrpcApiError: Failed to execute gRPC AEDT command: DetachFaces

In [76]:



	############ Subract barrier from rotor's iron ###############
barriers=""
PMs=""
temp=0


In [77]:
for jj in range (2):
	for ii in range(geo["nlay"]+geo["radial_ribs_split"]):
		rotorindex=(3+5*geo["q"]*3)*(1-jj)+ii+temp*jj
		barrier="2_%d" %(rotorindex)
		barriers=barriers+","+barrier
		
		

	for kk in range(int(geo["n_PM"]/2)):
		rotorindex=rotorindex+1
		PM="2_%d" %(rotorindex)
		PMs=PMs+","+PM

	temp=rotorindex+1
#oDesktop.AddMessage ("%s"%(geo["filename"][:-4]), "Maxwell2DDesign1", 0, "%s"%(barriers), "")


In [78]:
barriers=barriers[1:] 
barriers_array = barriers.split(",")
PMs=PMs[1:]
PMs_array=PMs.split(',')

rotorplate="2_%d" %(temp)
shaft="2_%d" %(temp+1)


In [79]:

m2dEditor.Subtract(
	[
		"NAME:Selections",
		"Blank Parts:="		, rotorplate,
		"Tool Parts:="		, barriers
	], 
	[
		"NAME:SubtractParameters",
		"KeepOriginals:="	, False
	])

if geo["n_PM"]!=0:
	m2dEditor.Subtract(
		[
			"NAME:Selections",
			"Blank Parts:="		, rotorplate,
			"Tool Parts:="		, PMs
		], 
		[
			"NAME:SubtractParameters",
			"KeepOriginals:="	, True
		])


In [80]:

################### Material Assignment ####################
###Rotor
m2dEditor.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Geometry3DAttributeTab",
			[
				"NAME:PropServers",rotorplate
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Solve Inside","Value:="		, True
				],
				[
					"NAME:Color","R:=", 140,"G:=", 160,	"B:=", 175
				],
				[
					"NAME:Material",
					"Value:="		, "\"%s\"" %(material["rotor"])
				]
			]
		]
	])



In [81]:


###Stator
m2dEditor.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Geometry3DAttributeTab",
			[
				"NAME:PropServers",statplate
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Solve Inside","Value:="		, True
				],
				[
					"NAME:Color","R:=", 140,"G:=", 160,	"B:=", 175
				],
				[
					"NAME:Material",
					"Value:="		, "\"%s\"" %(material["stator"])
				]
			]
		]
	])
	


In [82]:
###Shaft
m2dEditor.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Geometry3DAttributeTab",
			[
				"NAME:PropServers",shaft
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Solve Inside","Value:="		, True
				],
				[
					"NAME:Color","R:=", 140,"G:=", 160,	"B:=", 175
				],
				[
					"NAME:Material",
					"Value:="		, "\"%s\"" %(material["shaft"])
				]
			]
		]
	])


In [84]:

m2dEditor.SetWCS(["NAME:SetWCS Parameter","Working Coordinate System:=", "Global","RegionDepCSOk:=", False])
###Magnet
if geo["n_PM"]!=0:
	
	for ii in range(len(PMs_array)):
		m2dEditor.CreateRelativeCS(
		[
			"NAME:RelativeCSParameters",
			"Mode:="		, "Axis/Position",
			"OriginX:="		, "%smm"%(geo["PM_CS"][0][ii]),
			"OriginY:="		, "%smm"%(geo["PM_CS"][1][ii]),
			"OriginZ:="		, "0mm",
			"XAxisXvec:="		, "%smm"%(geo["PM_CS"][2][ii]),
			"XAxisYvec:="		, "%smm"%(geo["PM_CS"][3][ii]),
			"XAxisZvec:="		, "0mm",
			"YAxisXvec:="		, "%smm"%(-geo["PM_CS"][3][ii]),
			"YAxisYvec:="		, "%smm"%(geo["PM_CS"][2][ii]),
			"YAxisZvec:="		, "0mm"
		], 
		[
			"NAME:Attributes",
			"Name:="		, "RelativeCS%s"%(PMs_array[ii])
		])

		m2dEditor.ChangeProperty(
			[
				"NAME:AllTabs",
				[
					"NAME:Geometry3DAttributeTab",
					[
						"NAME:PropServers",PMs_array[ii]
					],
					[
						"NAME:ChangedProps",
						[
							"NAME:Solve Inside","Value:="		, True
						],
						[
							"NAME:Color","R:=", 50,"G:=", 50,	"B:=", 50
						],
						[
							"NAME:Material",
							"Value:="		, "\"%s\"" %(material["magnet"])
						],
						[
							"NAME:Orientation",
							"Value:="		, "RelativeCS%s"%(PMs_array[ii])
						]
					]
				]
			])
		m2dEditor.SetWCS(["NAME:SetWCS Parameter","Working Coordinate System:=", "Global","RegionDepCSOk:=", False])


In [85]:
###Conductor
for ii in range(len(slotconds_array)): ##3 fasi
	m2dEditor.ChangeProperty(
		[
			"NAME:AllTabs",
			[
				"NAME:Geometry3DAttributeTab",
				[
					"NAME:PropServers",slotconds_array[ii]
				],
				[
					"NAME:ChangedProps",
					[
						"NAME:Solve Inside","Value:="		, True
					],
					[
						"NAME:Color","R:=", 220,"G:=", 165,	"B:=", 30
					],
					[
						"NAME:Material",
						"Value:="		, "\"%s\"" %(material["slotcond"])
					]
				]
			]
	])

# barrier removed because air background, programm more stable
# ###Aria: Barriere di flusso
# for ii in range(len(barriers_array)):
# 	m2dEditor.ChangeProperty(
# 		[
# 			"NAME:AllTabs",
# 			[
# 				"NAME:Geometry3DAttributeTab",
# 				[
# 					"NAME:PropServers",barriers_array[ii] 
# 				],
# 				[
# 					"NAME:ChangedProps",
# 					[
# 						"NAME:Solve Inside","Value:="		, True
# 					],
# 					[
# 						"NAME:Color","R:=", 80,"G:=", 150,	"B:=", 250
# 					],
# 					[
# 						"NAME:Material",
# 						"Value:="		, "\"Air\""
# 					]
# 				]
# 			]
# 		])


In [87]:
slotairgaps_array

['1_1_Detach1',
 '1_1_Detach2',
 '1_1_Detach3',
 '1_1_Detach4',
 '1_1_Detach5',
 '1_1_Detach6',
 '1_1_Detach7',
 '1_1_Detach8',
 '1_1_Detach9']

In [88]:
len(slotairgaps_array)

9

In [89]:
range(len(slotairgaps_array))

range(0, 9)

In [91]:
ii=0

In [90]:

###Aria: Traferro di cava
for ii in range(len(slotairgaps_array)):
	m2dEditor.ChangeProperty(
		[
			"NAME:AllTabs",
			[
				"NAME:Geometry3DAttributeTab",
				[
					"NAME:PropServers",slotairgaps_array[ii]  
				],
				[
					"NAME:ChangedProps",
					[
						"NAME:Solve Inside","Value:="		, True
					],
					[
						"NAME:Color","R:=", 80,"G:=", 150,	"B:=", 250
					],
					[
						"NAME:Material",
						"Value:="		, "\"Air\""
					]
				]
			]
		])

##clen warning due to solve inside
# oDesktop=m2d.odesktop
# oDesktop.
# ("", "",2)


GrpcApiError: Failed to execute gRPC AEDT command: ChangeProperty

In [98]:

############################# Boundaries Geometry ###############################
#Circular Region Conteining Motor
m2dEditor.CreateCircle(
	[
		"NAME:CircleParameters","IsCovered:=", True,"XCenter:=", "0mm","YCenter:=", "0mm","ZCenter:=", "0mm","Radius:=", "%smm"%(geo["R"]),"WhichAxis:=", "Z","NumSegments:=", "0"
	], 
	[
		"NAME:Attributes","Name:=", "Region","Flags:=", "",	"Color:=", "(70 180 250)","Transparency:="	, 0.85,"PartCoordinateSystem:=","Global","UDMId:=", "","MaterialValue:=", "\"Air\"",
		"SurfaceMaterialValue:=", "\"\"","SolveInside:=", True,	"IsMaterialEditable:=", True,"UseMaterialAppearance:=", False,"IsLightweight:="	, False
	])
#Master Boundary (lower segment)
m2dEditor.CreatePolyline(
	["NAME:PolylineParameters","IsPolylineCovered:=", True,"IsPolylineClosed:="	, False,
		["NAME:PolylinePoints",["NAME:PLPoint","X:=", "0mm","Y:=", "0mm","Z:=", "0mm"],
			["NAME:PLPoint","X:=", "%smm"%(geo["R"]),"Y:=", "0mm","Z:=", "0mm"]
		],
		["NAME:PolylineSegments",["NAME:PLSegment","SegmentType:=", "Line","StartIndex:=", 0,"NoOfPoints:=",2]],
		["NAME:PolylineXSection","XSectionType:=", "None","XSectionOrient:=", "Auto","XSectionWidth:=", "0mm","XSectionTopWidth:=", "0mm","XSectionHeight:=", "0mm","XSectionNumSegments:="	, "0","XSectionBendType:="	, "Corner"]
	], 
	[
		"NAME:Attributes","Name:=", "Boundary_Master","Flags:=", "","Color:=","(143 175 143)","Transparency:=", 0,"PartCoordinateSystem:=", "Global",	"UDMId:=", "","MaterialValue:=", "\"vacuum\"",
		"SurfaceMaterialValue:=", "\"\"","SolveInside:=", True,	"IsMaterialEditable:=", True,"UseMaterialAppearance:=", False,"IsLightweight:="	, False
	])

#Slave Boundary (vertical segment)

motorangle=math.pi/geo["p"]
m2dEditor.CreatePolyline(
	["NAME:PolylineParameters","IsPolylineCovered:=", True,"IsPolylineClosed:="	, False,
		["NAME:PolylinePoints",["NAME:PLPoint","X:=", "0mm","Y:=", "0mm","Z:=", "0mm"],
			["NAME:PLPoint","X:=", "%smm"%(geo["R"]*math.cos(motorangle)),"Y:=", "%smm"%(geo["R"]*math.sin(motorangle)),"Z:=", "0mm"]
		],
		["NAME:PolylineSegments",["NAME:PLSegment","SegmentType:=", "Line","StartIndex:=", 0,"NoOfPoints:=",2]],
		["NAME:PolylineXSection","XSectionType:=", "None","XSectionOrient:=", "Auto","XSectionWidth:=", "0mm","XSectionTopWidth:=", "0mm","XSectionHeight:=", "0mm","XSectionNumSegments:="	, "0","XSectionBendType:="	, "Corner"]
	], 
	[
		"NAME:Attributes","Name:=", "Boundary_Slave","Flags:=", "","Color:=","(143 175 143)","Transparency:=", 0,"PartCoordinateSystem:=", "Global",	"UDMId:=", "","MaterialValue:=", "\"vacuum\"",
		"SurfaceMaterialValue:=", "\"\"","SolveInside:=", True,	"IsMaterialEditable:=", True,"UseMaterialAppearance:=", False,"IsLightweight:="	, False
	])
#Outer Stator Arc, 0 vector potenzial
m2dEditor.CreatePolyline(
	["NAME:PolylineParameters","IsPolylineCovered:=",True,"IsPolylineClosed:=",False,["NAME:PolylinePoints",
			["NAME:PLPoint","X:=","%smm"%(geo["R"]),"Y:=","0mm","Z:=", "0mm"], #primo pt arco
			["NAME:PLPoint","X:=", "%smm"%(geo["R"]*math.cos(motorangle/2)),"Y:=", "%smm"%(geo["R"]*math.sin(motorangle/2)),"Z:=", "0mm"], #pt centale arco
			["NAME:PLPoint","X:=","%smm"%(geo["R"]*math.cos(motorangle)),"Y:=", "%smm"%(geo["R"]*math.sin(motorangle)),"Z:=","0mm"]#ultimo pt arco
		],
		["NAME:PolylineSegments",["NAME:PLSegment","SegmentType:=", "AngularArc","StartIndex:=", 0,	"NoOfPoints:=", 3,"NoOfSegments:=", "0",
				"ArcAngle:=", "%srad"%(motorangle),"ArcCenterX:=", "0mm","ArcCenterY:=","0mm","ArcCenterZ:=", "0mm","ArcPlane:=","XY"]],
		["NAME:PolylineXSection","XSectionType:="	, "None","XSectionOrient:="	, "Auto","XSectionWidth:=","0mm","XSectionTopWidth:=","0mm",
			"XSectionHeight:=","0mm","XSectionNumSegments:=","0","XSectionBendType:=","Corner"]
	], 
	["NAME:Attributes","Name:=","VectorPotential1","Flags:=","","Color:=","(143 175 143)","Transparency:=", 0,"PartCoordinateSystem:=", "Global",
		"UDMId:=","","MaterialValue:=", "\"vacuum\"","SurfaceMaterialValue:=", "\"\"","SolveInside:=", True,"IsMaterialEditable:=", True,
		"UseMaterialAppearance:=", False,"IsLightweight:=", False
	])

##Region(sector) conteining moving parts (rotor)
r_mov_mid=geo["r"]+geo["g"]/2 #raggio regione mid
r_mov_out=geo["r"]+geo["g"]*3/4 #raggio regione out

#circuilar region outer rotor
m2dEditor.CreateCircle(
	[
		"NAME:CircleParameters","IsCovered:=", True,"XCenter:=", "0mm","YCenter:=", "0mm","ZCenter:=", "0mm","Radius:=", "%smm"%(r_mov_out),"WhichAxis:=", "Z","NumSegments:=", "0"
	], 
	[
		"NAME:Attributes","Name:=", "Rotating_band_out","Flags:=", "",	"Color:=", "(70 180 250)","Transparency:="	, 0.85,"PartCoordinateSystem:=","Global","UDMId:=", "","MaterialValue:=", "\"Air\"",
		"SurfaceMaterialValue:=", "\"\"","SolveInside:=", True,	"IsMaterialEditable:=", True,"UseMaterialAppearance:=", False,"IsLightweight:="	, False
	])

#cut circular region
m2dEditor.Imprint(["NAME:Selections","Blank Parts:=","Region,Rotating_band_out","Tool Parts:=", "Boundary_Master,Boundary_Slave"	],["NAME:ImprintParameters","KeepOriginals:=",True])

facesIDs=m2dEditor.GetFaceIDs("Region")
faceID=int(facesIDs[1])
m2dEditor.DetachFaces(["NAME:Selections",	"Selections:=","Region","NewPartsModelFlag:=","Model"],["NAME:Parameters",["NAME:DetachFacesToParameters","FacesToDetach:=", [faceID]]])
m2dEditor.Delete(["NAME:Selections","Selections:=","Region_Detach1"])

facesIDs=m2dEditor.GetFaceIDs("Rotating_band_out")
faceID=int(facesIDs[1])
m2dEditor.DetachFaces(["NAME:Selections",	"Selections:=","Rotating_band_out","NewPartsModelFlag:=","Model"],["NAME:Parameters",["NAME:DetachFacesToParameters","FacesToDetach:=", [faceID]]])
m2dEditor.Delete(["NAME:Selections","Selections:=","Rotating_band_out_Detach1"])

#circuilar region mid airgap 
m2dEditor.CreateCircle(
	[
		"NAME:CircleParameters","IsCovered:=", True,"XCenter:=", "0mm","YCenter:=", "0mm","ZCenter:=", "0mm","Radius:=", "%smm"%(r_mov_mid),"WhichAxis:=", "Z","NumSegments:=", "0"
	], 
	[
		"NAME:Attributes","Name:=", "Rotating_band_mid","Flags:=", "",	"Color:=", "(70 180 250)","Transparency:="	, 0.85,"PartCoordinateSystem:=","Global","UDMId:=", "","MaterialValue:=", "\"Air\"",
		"SurfaceMaterialValue:=", "\"\"","SolveInside:=", True,	"IsMaterialEditable:=", True,"UseMaterialAppearance:=", False,"IsLightweight:="	, False
	])

#cut circular region
m2dEditor.Imprint(["NAME:Selections","Blank Parts:=","Rotating_band_mid","Tool Parts:=", "Boundary_Master,Boundary_Slave"	],["NAME:ImprintParameters","KeepOriginals:=",True])

